# 作业 04：正则表达式与结构化网页数据

**学号**：20244080326　　**姓名**：罗娜

本作业包含：

1. 12 道 Python 正则表达式练习，每题均附测试用例与断言；
2. 使用 Requests 和 BeautifulSoup 从城市网页定位气候表格，并提取行名。

In [1]:
import hashlib
import os
import platform
import sys
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080326
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV") or os.path.basename(sys.prefix)
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹]   {MACHINE_FP}")
print(f"[RUN_ID]     {RUN_ID}")
print(f"[Python]     {sys.version.split()[0]}")


[Conda 环境] conda-hw
[学号]       20244080326
[机器指纹]   e8e3a7d29233976b
[RUN_ID]     20244080326-e8e3a7d29233976b
[Python]     3.11.17


## 一、Python 正则表达式练习

以下“匹配”题均使用 `re.fullmatch()`，要求整个字符串符合规则，而不是只匹配其中一段。

### 1. 检查字符串是否仅包含英文字母和数字

In [2]:
import re


def is_alphanumeric(text):
    """非空且仅含 a-z、A-Z、0-9 时返回 True。"""
    return re.fullmatch(r"[A-Za-z0-9]+", text) is not None


tests = [("Hello123", True), ("abcXYZ09", True), ("hello_world", False),
         ("hello world", False), ("中文123", False), ("", False)]
for text, expected in tests:
    result = is_alphanumeric(text)
    print(f"{text!r} -> {result}")
    assert result == expected

'Hello123' -> True
'abcXYZ09' -> True
'hello_world' -> False
'hello world' -> False
'中文123' -> False
'' -> False


### 2. 匹配一个 a 后面跟着零个或多个 b 的字符串

In [3]:
def match_a_zero_or_more_b(text):
    return re.fullmatch(r"ab*", text) is not None


tests = [("a", True), ("ab", True), ("abbbb", True),
         ("b", False), ("ac", False), ("ba", False)]
for text, expected in tests:
    result = match_a_zero_or_more_b(text)
    print(f"{text!r} -> {result}")
    assert result == expected

'a' -> True
'ab' -> True
'abbbb' -> True
'b' -> False
'ac' -> False
'ba' -> False


### 3. 匹配一个 a 后面跟着一个或多个 b 的字符串

In [4]:
def match_a_one_or_more_b(text):
    return re.fullmatch(r"ab+", text) is not None


tests = [("ab", True), ("abb", True), ("abbbb", True),
         ("a", False), ("b", False), ("abc", False)]
for text, expected in tests:
    result = match_a_one_or_more_b(text)
    print(f"{text!r} -> {result}")
    assert result == expected

'ab' -> True
'abb' -> True
'abbbb' -> True
'a' -> False
'b' -> False
'abc' -> False


### 4. 匹配以 a 开头、以 b 结尾，中间可有任意字符的字符串

In [5]:
def match_a_to_b(text):
    return re.fullmatch(r"a.*b", text) is not None


tests = [("ab", True), ("a123b", True), ("a hello b", True),
         ("a", False), ("b", False), ("axc", False), ("xab", False)]
for text, expected in tests:
    result = match_a_to_b(text)
    print(f"{text!r} -> {result}")
    assert result == expected

'ab' -> True
'a123b' -> True
'a hello b' -> True
'a' -> False
'b' -> False
'axc' -> False
'xab' -> False


### 5. 删除 IP 地址各段数字的前导零

In [6]:
def remove_ip_leading_zeros(ip):
    """把每段数字转成 int 再转回字符串，全零段自然保留为一个 0。"""
    return re.sub(r"\d+", lambda match: str(int(match.group())), ip)


tests = [("192.168.001.010", "192.168.1.10"),
         ("000.000.010.000", "0.0.10.0"),
         ("10.0.0.1", "10.0.0.1")]
for ip, expected in tests:
    result = remove_ip_leading_zeros(ip)
    print(f"{ip} -> {result}")
    assert result == expected

192.168.001.010 -> 192.168.1.10
000.000.010.000 -> 0.0.10.0
10.0.0.1 -> 10.0.0.1


### 6. 将空格替换为下划线，并将下划线替换为空格

In [7]:
def swap_spaces_underscores(text):
    """一次替换完成双向交换，避免第二次替换覆盖第一次的结果。"""
    return re.sub(r"[ _]", lambda m: "_" if m.group() == " " else " ", text)


tests = [("hello world_test", "hello_world test"),
         ("a_b c_d", "a b_c d"),
         ("no-change", "no-change")]
for text, expected in tests:
    result = swap_spaces_underscores(text)
    print(f"{text!r} -> {result!r}")
    assert result == expected

'hello world_test' -> 'hello_world test'
'a_b c_d' -> 'a b_c d'
'no-change' -> 'no-change'


### 7. 将 yyyy-mm-dd 日期转换为 dd-mm-yyyy

In [8]:
def convert_date(text):
    return re.sub(r"\b(\d{4})-(\d{2})-(\d{2})\b", r"\3-\2-\1", text)


tests = [("2026-11-26", "26-11-2026"),
         ("开始日期：2024-05-01", "开始日期：01-05-2024"),
         ("2024-05-01 至 2024-05-03", "01-05-2024 至 03-05-2024")]
for text, expected in tests:
    result = convert_date(text)
    print(f"{text} -> {result}")
    assert result == expected

2026-11-26 -> 26-11-2026
开始日期：2024-05-01 -> 开始日期：01-05-2024
2024-05-01 至 2024-05-03 -> 01-05-2024 至 03-05-2024


### 8. 找出所有以 a 或 e 开头的单词

In [9]:
def find_words_starting_ae(text):
    return re.findall(r"\b[ae][A-Za-z]*\b", text, flags=re.IGNORECASE)


tests = [("apple banana egg Elephant ant", ["apple", "egg", "Elephant", "ant"]),
         ("An orange and an umbrella", ["An", "and", "an"]),
         ("sky blue", [])]
for text, expected in tests:
    result = find_words_starting_ae(text)
    print(f"{text!r} -> {result}")
    assert result == expected

'apple banana egg Elephant ant' -> ['apple', 'egg', 'Elephant', 'ant']


'An orange and an umbrella' -> ['An', 'and', 'an']
'sky blue' -> []


### 9. 将字符串中的 Road 缩写为 Rd

In [10]:
def abbreviate_road(text):
    return re.sub(r"\bRoad\b", "Rd", text)


tests = [("Road 1 and Broadway Road", "Rd 1 and Broadway Rd"),
         ("Roadside is not Road", "Roadside is not Rd"),
         ("21 Road", "21 Rd")]
for text, expected in tests:
    result = abbreviate_road(text)
    print(f"{text!r} -> {result!r}")
    assert result == expected

'Road 1 and Broadway Road' -> 'Rd 1 and Broadway Rd'
'Roadside is not Road' -> 'Roadside is not Rd'
'21 Road' -> '21 Rd'


### 10. 找出所有长度至少为 4 个字符的单词

In [11]:
def find_long_words(text):
    return re.findall(r"\b[A-Za-z]{4,}\b", text)


tests = [("I like Python and data analysis", ["like", "Python", "data", "analysis"]),
         ("a an the tree code", ["tree", "code"]),
         ("a I an", [])]
for text, expected in tests:
    result = find_long_words(text)
    print(f"{text!r} -> {result}")
    assert result == expected

'I like Python and data analysis' -> ['like', 'Python', 'data', 'analysis']
'a an the tree code' -> ['tree', 'code']
'a I an' -> []


### 11. 删除除字母和数字之外的所有字符

In [12]:
def remove_non_alphanumeric(text):
    return re.sub(r"[^A-Za-z0-9]+", "", text)


tests = [("Hello, World_2026!", "HelloWorld2026"),
         ("a-b_c@example.com", "abcexamplecom"),
         ("中文 ABC 123", "ABC123")]
for text, expected in tests:
    result = remove_non_alphanumeric(text)
    print(f"{text!r} -> {result!r}")
    assert result == expected

'Hello, World_2026!' -> 'HelloWorld2026'
'a-b_c@example.com' -> 'abcexamplecom'
'中文 ABC 123' -> 'ABC123'


### 12. 删除括号及括号内的内容

In [13]:
def remove_parentheses(text):
    return re.sub(r"\s*\([^()]*\)", "", text).strip()


inputs = ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]
expected = ["example", "w3resource", "github", "stackoverflow"]
results = [remove_parentheses(text) for text in inputs]
for original, result in zip(inputs, results):
    print(f"{original!r} -> {result!r}")
assert results == expected
print()
print("12 道正则题的全部断言均通过。")

'example (.com)' -> 'example'
'w3resource' -> 'w3resource'
'github (.com)' -> 'github'
'stackoverflow (.com)' -> 'stackoverflow'

12 道正则题的全部断言均通过。


## 二、从网页检索气候数据

**实际访问日期：2026-11-26。**

按题目建议，先尝试了 Wikipedia 的 Madison、Harbin、Los Angeles 页面，但当前网络出现连接重置或超时。根据题目“遇到访问错误应更换可正常访问且包含气候数据表格的页面”的要求，实际改用：

- Madison：https://www.weather2travel.com/wisconsin/madison/climate/
- Los Angeles：https://www.weather2travel.com/california/los-angeles/climate/
- Boston：https://usclimatedata.com/climate/boston/massachusetts/united-states/usma0046

前两个页面将气候指标放在行名、月份放在列标题；Boston 页面将月份放在行名，并使用 `scope="row"`，可用于测试结构差异。所有网络请求均由下面的代码实时完成。

### 1. 读取 HTML、解析并观察气候表格结构

辅助函数 `_read_soup()` 只负责读取和解析；HTTP 错误通过 `raise_for_status()` 抛出，连接错误等 `requests` 异常也不捕获，因此不会被误报为“没有表格”。

In [14]:
from datetime import date
import re
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag

ACCESS_DATE = date.today().isoformat()
MADISON_URL = "https://www.weather2travel.com/wisconsin/madison/climate/"
LOS_ANGELES_URL = "https://www.weather2travel.com/california/los-angeles/climate/"
BOSTON_URL = "https://usclimatedata.com/climate/boston/massachusetts/united-states/usma0046"
NO_CLIMATE_URL = "https://example.com/"

MONTHS = {
    "jan", "january", "feb", "february", "mar", "march", "apr", "april",
    "may", "jun", "june", "jul", "july", "aug", "august", "sep", "sept",
    "september", "oct", "october", "nov", "november", "dec", "december"
}


def clean_text(value):
    """合并连续空白并去除首尾空白。"""
    return re.sub(r"\s+", " ", value).strip()


def read_soup(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; DataAnalysisHomework/1.0)"},
        timeout=45,
    )
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def looks_like_climate_table(table):
    """识别 Wikipedia 标题，或具有完整月份轴和气候指标的月度表格。"""
    table_text = clean_text(table.get_text(" ", strip=True)).lower()
    if "climate data for" in table_text:
        return True

    rows = table.find_all("tr")
    month_columns = 0
    for row in rows[:3]:
        texts = [clean_text(c.get_text(" ", strip=True)).lower().rstrip(".")
                 for c in row.find_all(["th", "td"])]
        month_columns = max(month_columns, sum(text in MONTHS for text in texts))

    month_rows = 0
    for row in rows[1:]:
        cells = row.find_all(["th", "td"], recursive=False)
        if cells:
            first = clean_text(cells[0].get_text(" ", strip=True)).lower().rstrip(".")
            month_rows += first in MONTHS

    climate_words = ("temperature", "precip", "rainfall", "humidity",
                     "high", "low", "climate")
    has_climate_words = any(word in table_text for word in climate_words)
    return (month_columns >= 6 or month_rows >= 6) and has_climate_words


print("访问日期：", ACCESS_DATE)
madison_soup = read_soup(MADISON_URL)
madison_table = next(
    (table for table in madison_soup.find_all("table") if looks_like_climate_table(table)),
    None,
)
assert madison_table is not None
children = [child.name for child in madison_table.children if getattr(child, "name", None)]
print("Madison 表格直接子节点：", children)
print("直接子节点数量：", len(children))
print("表格行数：", len(madison_table.find_all("tr")))
for row in madison_table.find_all("tr")[:3]:
    print([clean_text(cell.get_text(" ", strip=True))
           for cell in row.find_all(["th", "td"], recursive=False)])

访问日期： 2026-11-26


Madison 表格直接子节点： ['thead', 'tbody']
直接子节点数量： 2
表格行数： 4
['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
['Max temperature (daytime)', '-4', '-1', '5', '14', '21', '26', '28', '27', '22', '16', '7', '-1']
['Min temperature (night-time)', '-14', '-12', '-5', '1', '7', '12', '15', '14', '9', '3', '-3', '-11']


Madison 的目标 `<table>` 有两个直接子节点：`<thead>` 和 `<tbody>`。第一行的 `<th>` 是月份列标题；后续行由一个行名 `<th>` 和若干数据 `<td>` 组成。

### 2. 编写 retrieve_climate_table(s)

In [15]:
def retrieve_climate_table(s):
    """返回 URL 页面中的气候表格 Tag；无表格返回 None；网络/HTTP 错误抛出异常。"""
    soup = read_soup(s)
    return next(
        (table for table in soup.find_all("table") if looks_like_climate_table(table)),
        None,
    )


for url in [MADISON_URL, LOS_ANGELES_URL, BOSTON_URL]:
    table = retrieve_climate_table(url)
    assert isinstance(table, Tag)
    children = [child.name for child in table.children if getattr(child, "name", None)]
    print(url)
    print("  类型：", type(table).__name__, "；行数：", len(table.find_all("tr")),
          "；直接子节点：", children)

assert retrieve_climate_table(NO_CLIMATE_URL) is None
print("无气候表格的页面正确返回 None")

try:
    retrieve_climate_table("https://example.invalid/")
except requests.RequestException as exc:
    print("网络错误按预期向外抛出：", type(exc).__name__)
else:
    raise AssertionError("无效域名应触发网络异常")

https://www.weather2travel.com/wisconsin/madison/climate/
  类型： Tag ；行数： 4 ；直接子节点： ['thead', 'tbody']


https://www.weather2travel.com/california/los-angeles/climate/
  类型： Tag ；行数： 4 ；直接子节点： ['thead', 'tbody']


https://usclimatedata.com/climate/boston/massachusetts/united-states/usma0046
  类型： Tag ；行数： 13 ；直接子节点： ['thead', 'tbody']


无气候表格的页面正确返回 None
网络错误按预期向外抛出： ConnectionError


三个城市页面均返回 `bs4.element.Tag`，无气候表格的页面返回 `None`，无效域名则抛出网络异常。

结构差异：Madison 与 Los Angeles 的表格以气候指标为行名；Boston 的表格以月份为行名，并在这些行名单元格上使用 `scope="row"`。

### 3. 编写 list_climate_table_row_names(s)

提取规则：

1. 优先使用 `scope="row"` 的 `<th>`，否则使用同一行中其他非 `scope="col"` 的 `<th>`；
2. 没有 `<td>` 的标题行、月份列标题行和来源行直接跳过；
3. 清理标题、来源、月份列标题，所有结果均为去首尾空白的 Python 字符串。

In [16]:
def list_climate_table_row_names(s):
    """返回气候表格行名字符串列表；无气候表格时返回 None。"""
    table = retrieve_climate_table(s)
    if table is None:
        return None

    row_names = []
    for row in table.find_all("tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if not cells or not row.find("td", recursive=False):
            continue

        texts = [clean_text(cell.get_text(" ", strip=True)) for cell in cells]
        joined = " ".join(texts).lower()
        if "climate data for" in joined or joined.startswith(("source", "data source")):
            continue
        if sum(text.lower().rstrip(".") in MONTHS for text in texts) >= 2:
            continue

        header = next(
            (cell for cell in cells
             if cell.name == "th" and cell.get("scope", "").lower() == "row"),
            None,
        )
        if header is None:
            header = next(
                (cell for cell in cells
                 if cell.name == "th" and cell.get("scope", "").lower() != "col"),
                None,
            )
        if header is None:
            continue

        name = clean_text(header.get_text(" ", strip=True))
        if name and name.lower() != "month" and not name.lower().startswith("source"):
            row_names.append(name)
    return row_names


names_by_url = {}
for url in [MADISON_URL, LOS_ANGELES_URL, BOSTON_URL]:
    names = list_climate_table_row_names(url)
    names_by_url[url] = names
    assert names and all(isinstance(name, str) for name in names)
    assert all(name == name.strip() for name in names)
    print(url)
    print(names)

assert names_by_url[MADISON_URL][0] == "Max temperature (daytime)"
assert not any(name in {"Jan", "Feb", "Dec"} for name in names_by_url[MADISON_URL])
assert names_by_url[BOSTON_URL] == [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]
assert list_climate_table_row_names(NO_CLIMATE_URL) is None
print("行名提取与无表格分支的全部断言均通过。")

https://www.weather2travel.com/wisconsin/madison/climate/
['Max temperature (daytime)', 'Min temperature (night-time)', 'Heat & Humidity']


https://www.weather2travel.com/california/los-angeles/climate/
['Max temperature (daytime)', 'Min temperature (night-time)', 'Heat & Humidity']


https://usclimatedata.com/climate/boston/massachusetts/united-states/usma0046
['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


行名提取与无表格分支的全部断言均通过。


**结果说明**

- Madison、Los Angeles 返回的是“Max temperature”“Min temperature”等气候指标，月份列标题被排除；
- Boston 的结构不同，月份本身就是行名，因此返回 `Jan` 至 `Dec`，列标题 `Month` 不计入；
- 标题行和数据来源行均未进入结果，列表元素全部是普通 Python 字符串。